# PC-RGNN: nuScenes mini object-level detection development track

This is a separate track from the PCL integrity demonstration. It uses the official `v1.0-mini` nuScenes tables and LiDAR files when the user mounts them locally. Raw nuScenes data is intentionally not committed. The model is an object-level development detector built from jittered proposal crops with ground-truth box targets; it is not a full-scene proposal system and its mini-split metrics are not official leaderboard results.

## Google Colab setup

Run this notebook from top to bottom in Colab. The next cell installs the official `nuscenes-devkit` from PyPI. The dataset itself is not downloaded by pip: download the official `v1.0-mini` archive from the nuScenes account page, upload/extract it in Colab or mount Google Drive, then set `NUSCENES_ROOT` to the folder containing `v1.0-mini`. Raw dataset files remain external to this repository.

In [1]:
# Colab dependency and data-path setup. Safe to rerun.
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nuscenes-devkit'])
    print('Installed nuscenes-devkit in this Colab runtime.')
    print('Mount Drive or upload/extract v1.0-mini, then set NUSCENES_ROOT below if needed.')
else:
    print('Local runtime detected; install with: python -m pip install nuscenes-devkit')

# Examples: /content/nuScenes, /content/drive/MyDrive/nuScenes, or a local project data folder.
if 'NUSCENES_ROOT' not in os.environ:
    os.environ['NUSCENES_ROOT'] = '/content/nuScenes' if IN_COLAB else str(Path.cwd() / 'data' / 'nuscenes')
print('NUSCENES_ROOT:', os.environ['NUSCENES_ROOT'])

Local runtime detected; install with: python -m pip install nuscenes-devkit
NUSCENES_ROOT: /home/aliaqil/Desktop/projects/pcrgnn/data/nuscenes


## Data contract

Set `NUSCENES_ROOT` to either the directory containing the downloaded nuScenes mini release or its `v1.0-mini` table directory. The notebook uses the official devkit for scenes, categories, poses, boxes, and LiDAR. Train and validation rows are separated by scene token. In Colab, run the setup cell first, mount/upload the official archive, and point `NUSCENES_ROOT` at the extracted folder. If the dataset is not mounted, data-dependent cells skip with setup instructions rather than fabricate results.

In [2]:
import os
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIGURED_NUSCENES_ROOT = Path(os.environ.get('NUSCENES_ROOT', PROJECT_ROOT / 'data' / 'nuscenes'))
candidate_version_roots = [CONFIGURED_NUSCENES_ROOT / 'v1.0-mini', CONFIGURED_NUSCENES_ROOT]
VERSION_ROOT = next((path for path in candidate_version_roots if (path / 'scene.json').exists() and (path / 'sample.json').exists()), candidate_version_roots[0])
NUSCENES_ROOT = VERSION_ROOT.parent if VERSION_ROOT.name == 'v1.0-mini' else CONFIGURED_NUSCENES_ROOT
DATA_AVAILABLE = (VERSION_ROOT / 'scene.json').exists() and (VERSION_ROOT / 'sample.json').exists()
print('configured NUSCENES_ROOT:', CONFIGURED_NUSCENES_ROOT)
print('resolved table root:', VERSION_ROOT)
if not DATA_AVAILABLE:
    print('nuScenes mini not mounted. Download v1.0-mini from the official nuScenes site, then set NUSCENES_ROOT to its parent directory or to v1.0-mini.')

configured NUSCENES_ROOT: /home/aliaqil/Desktop/projects/pcrgnn/data/nuscenes
resolved table root: /home/aliaqil/Desktop/projects/pcrgnn/data/nuscenes/v1.0-mini
nuScenes mini not mounted. Download v1.0-mini from the official nuScenes site, then set NUSCENES_ROOT to its parent directory or to v1.0-mini.


In [3]:
TORCH_AVAILABLE = False
try:
    import torch
    from torch import nn
    TORCH_AVAILABLE = True
except ImportError as exc:
    print('PyTorch unavailable:', exc)

NUSCENES_DEVKIT_AVAILABLE = False
try:
    from nuscenes.nuscenes import NuScenes
    from nuscenes.utils.data_classes import LidarPointCloud
    NUSCENES_DEVKIT_AVAILABLE = True
except ImportError as exc:
    print('nuScenes devkit unavailable:', exc)

DEVKIT_AVAILABLE = TORCH_AVAILABLE and NUSCENES_DEVKIT_AVAILABLE

if DATA_AVAILABLE and not DEVKIT_AVAILABLE:
    raise ImportError('Install nuscenes-devkit and PyTorch before running the mounted-data track.')
if not TORCH_AVAILABLE:
    class _MissingNN:
        Module = object
    nn = _MissingNN()

if DATA_AVAILABLE:
    nusc = NuScenes(version='v1.0-mini', dataroot=str(NUSCENES_ROOT), verbose=False)
    print('loaded official nuScenes version:', nusc.version, '| scenes:', len(nusc.scene))

In [4]:
CATEGORY_MAP = {
    'vehicle.car': 'car',
    'vehicle.truck': 'truck',
    'vehicle.bus.bendy': 'bus',
    'vehicle.bus.rigid': 'bus',
    'human.pedestrian.adult': 'pedestrian',
    'human.pedestrian.child': 'pedestrian',
    'human.pedestrian.construction_worker': 'pedestrian',
    'vehicle.bicycle': 'bicycle',
    'vehicle.motorcycle': 'motorcycle',
}
CLASS_NAMES = sorted(set(CATEGORY_MAP.values()))
CLASS_TO_ID = {name: index for index, name in enumerate(CLASS_NAMES)}

def split_scene_tokens(scene_records, validation_fraction=0.2):
    tokens = sorted(scene['token'] for scene in scene_records)
    cut = max(1, int(round(len(tokens) * (1 - validation_fraction))))
    return set(tokens[:cut]), set(tokens[cut:])

def crop_from_box(points, center, size, max_points=128, seed=0):
    rng = np.random.default_rng(seed)
    center = np.asarray(center, dtype=np.float32)
    size = np.asarray(size, dtype=np.float32)
    proposal_center = center + rng.normal(0.0, [0.5, 0.5, 0.2]).astype(np.float32)
    half = size * 0.75 + np.asarray([0.75, 0.75, 0.4], dtype=np.float32)
    mask = np.all(np.abs(points[:, :3] - proposal_center[None, :]) <= half[None, :], axis=1)
    crop = points[mask, :3] - proposal_center[None, :]
    if len(crop) == 0:
        return None
    indices = rng.choice(len(crop), size=max_points, replace=len(crop) < max_points)
    return crop[indices].astype(np.float32), (center - proposal_center).astype(np.float32)

def extract_object_crops(nusc, scene_tokens, max_points=128):
    rows = []
    for scene in nusc.scene:
        if scene['token'] not in scene_tokens:
            continue
        sample_token = scene['first_sample_token']
        while sample_token:
            sample = nusc.get('sample', sample_token)
            lidar_token = sample['data']['LIDAR_TOP']
            lidar_path, boxes, _ = nusc.get_sample_data(lidar_token)
            points = LidarPointCloud.from_file(lidar_path).points.T
            for box in boxes:
                class_name = CATEGORY_MAP.get(box.name)
                if class_name is None:
                    continue
                crop_result = crop_from_box(points, box.center, box.wlh, max_points=max_points, seed=len(rows))
                if crop_result is None:
                    continue
                crop, center_offset = crop_result
                yaw = float(box.orientation.yaw_pitch_roll[0])
                target = np.concatenate([center_offset, np.asarray(box.wlh, dtype=np.float32), [np.sin(yaw), np.cos(yaw)]])
                rows.append({'scene_token': scene['token'], 'sample_token': sample['token'], 'box_token': box.token, 'class_id': CLASS_TO_ID[class_name], 'class_name': class_name, 'points': crop, 'target': target.astype(np.float32)})
            sample_token = sample['next']
    return rows

if DATA_AVAILABLE:
    train_scenes, val_scenes = split_scene_tokens(nusc.scene)
    train_rows = extract_object_crops(nusc, train_scenes)
    val_rows = extract_object_crops(nusc, val_scenes)
    assert not ({row['scene_token'] for row in train_rows} & {row['scene_token'] for row in val_rows})
    print('train objects:', len(train_rows), '| validation objects:', len(val_rows), '| scene overlap: 0')
else:
    train_rows, val_rows = [], []

In [5]:
class GraphBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q = nn.Linear(channels, channels, bias=False)
        self.k = nn.Linear(channels, channels, bias=False)
        self.v = nn.Linear(channels, channels, bias=False)
        self.out = nn.Linear(channels, channels)

    def forward(self, x, indices):
        batch, points, channels = x.shape
        expanded = x[:, :, None, :].expand(-1, -1, indices.shape[-1], -1)
        gather = indices[None, :, :, None].expand(batch, -1, -1, channels)
        neighbours = torch.gather(expanded, 1, gather)
        score = (self.q(x)[:, :, None, :] * self.k(neighbours)).sum(-1) / channels**0.5
        weight = score.softmax(-1)
        update = (weight[..., None] * self.v(neighbours)).sum(2)
        return x + self.out(update)

class ObjectPCRGNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        channels = 48
        self.input = nn.Linear(3, channels)
        self.local = GraphBlock(channels)
        self.context = GraphBlock(channels)
        self.class_head = nn.Linear(channels, num_classes)
        self.box_head = nn.Linear(channels, 8)

    def forward(self, points, local_idx, context_idx):
        x = self.input(points)
        x = self.local(x, local_idx)
        x = self.context(x, context_idx)
        pooled = x.mean(1)
        return self.class_head(pooled), self.box_head(pooled)

def fixed_knn(points, k):
    from scipy.spatial import cKDTree
    return cKDTree(points).query(points, k=k + 1)[1][:, 1:]

def batch_from_rows(rows):
    points = np.stack([row['points'] for row in rows]).astype(np.float32)
    labels = np.asarray([row['class_id'] for row in rows], dtype=np.int64)
    targets = np.stack([row['target'] for row in rows]).astype(np.float32)
    local = np.stack([fixed_knn(point, 8) for point in points])
    context = np.stack([fixed_knn(point, 16) for point in points])
    return (torch.tensor(points), torch.tensor(labels), torch.tensor(targets), torch.tensor(local, dtype=torch.long), torch.tensor(context, dtype=torch.long))

def axis_aligned_iou_3d(center_a, size_a, center_b, size_b):
    lower = np.maximum(center_a - size_a / 2, center_b - size_b / 2)
    upper = np.minimum(center_a + size_a / 2, center_b + size_b / 2)
    intersection = np.prod(np.maximum(upper - lower, 0), axis=-1)
    volume_a = np.prod(size_a, axis=-1)
    volume_b = np.prod(size_b, axis=-1)
    return intersection / np.maximum(volume_a + volume_b - intersection, 1e-8)

def summarize_predictions(logits, predictions, labels, targets):
    pred_centers, pred_sizes = predictions[:, :3], np.maximum(predictions[:, 3:6], 1e-3)
    true_centers, true_sizes = targets[:, :3], targets[:, 3:6]
    pred_yaw = np.arctan2(predictions[:, 6], predictions[:, 7])
    true_yaw = np.arctan2(targets[:, 6], targets[:, 7])
    yaw_error = np.abs(np.arctan2(np.sin(pred_yaw - true_yaw), np.cos(pred_yaw - true_yaw)))
    iou = axis_aligned_iou_3d(pred_centers, pred_sizes, true_centers, true_sizes)
    assert np.isfinite(iou).all() and ((iou >= 0) & (iou <= 1)).all()
    assert np.isfinite(yaw_error).all()
    threshold_hits = (iou >= 0.25)
    # This oracle-crop evaluator has one candidate for each GT object; precision and recall are therefore diagnostic hit rates, not full-scene AP.
    hit_rate = float(threshold_hits.mean()) if len(iou) else 0.0
    return {'center_mae_m': float(np.abs(pred_centers - true_centers).mean()), 'size_mae_m': float(np.abs(pred_sizes - true_sizes).mean()), 'yaw_mae_rad': float(yaw_error.mean()), 'axis_aligned_iou3d_mean': float(iou.mean()), 'axis_aligned_iou3d_at_0_25': hit_rate, 'precision_at_0_25': hit_rate, 'recall_at_0_25': hit_rate}

print('model and metric definitions loaded; IoU is explicitly axis-aligned diagnostic IoU, not official oriented nuScenes AP.')

model and metric definitions loaded; IoU is explicitly axis-aligned diagnostic IoU, not official oriented nuScenes AP.


## Training process

For every training crop, the notebook masks a deterministic subset of the observed LiDAR points. The completion-aware model then: (1) encodes local and contextual k-neighborhoods with attention, (2) predicts missing points, (3) fuses observed and predicted points, and (4) predicts class and box parameters. The optimization objective is cross-entropy classification loss + Smooth L1 box loss + 0.25 × Chamfer completion loss. The training split contains complete scenes only; validation scenes are never used for optimization.

In [6]:
print('TRAINING PROCESS')
print('1. Load scene-disjoint training object crops')
print('2. Mask points to create partial LiDAR input and held-out completion targets')
print('3. Build local k=8 and contextual k=16 graphs')
print('4. Predict missing points, fuse them with observed points, and encode features')
print('5. Optimize classification + box regression + Chamfer completion loss for 5 epochs')
if not (DATA_AVAILABLE and DEVKIT_AVAILABLE and train_rows and val_rows):
    print('STATUS: training not run; mount nuScenes mini and install nuscenes-devkit to execute it.')

TRAINING PROCESS
1. Load scene-disjoint training object crops
2. Mask points to create partial LiDAR input and held-out completion targets
3. Build local k=8 and contextual k=16 graphs
4. Predict missing points, fuse them with observed points, and encode features
5. Optimize classification + box regression + Chamfer completion loss for 5 epochs
STATUS: training not run; mount nuScenes mini and install nuscenes-devkit to execute it.


In [7]:
class BatchedGraphBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q = nn.Linear(channels, channels, bias=False)
        self.k = nn.Linear(channels, channels, bias=False)
        self.v = nn.Linear(channels, channels, bias=False)
        self.out = nn.Linear(channels, channels)

    def forward(self, x, indices):
        batch, points, channels = x.shape
        if indices.ndim == 2:
            indices = indices[None].expand(batch, -1, -1)
        expanded = x[:, :, None, :].expand(-1, -1, indices.shape[-1], -1)
        gather = indices[..., None].expand(-1, -1, -1, channels)
        neighbours = torch.gather(expanded, 1, gather)
        score = (self.q(x)[:, :, None, :] * self.k(neighbours)).sum(-1) / channels**0.5
        weight = score.softmax(-1)
        return x + self.out((weight[..., None] * self.v(neighbours)).sum(2))

class CompletionAwareObjectPCRGNN(nn.Module):
    def __init__(self, num_classes, missing_points=32):
        super().__init__()
        channels = 48
        self.missing_points = missing_points
        self.input = nn.Linear(3, channels)
        self.local = BatchedGraphBlock(channels)
        self.context = BatchedGraphBlock(channels)
        self.completion_head = nn.Sequential(nn.Linear(channels, 64), nn.ReLU(), nn.Linear(64, missing_points * 3))
        self.completed_point_encoder = nn.Sequential(nn.Linear(3, channels), nn.ReLU(), nn.Linear(channels, channels))
        self.feature_head = nn.Sequential(nn.Linear(channels * 2, channels), nn.ReLU())
        self.class_head = nn.Linear(channels, num_classes)
        self.box_head = nn.Linear(channels, 8)

    def forward(self, points, local_idx, context_idx):
        encoded = self.input(points)
        encoded = self.local(encoded, local_idx)
        encoded = self.context(encoded, context_idx)
        observed_feature = encoded.mean(1)
        predicted_missing = self.completion_head(observed_feature).view(-1, self.missing_points, 3)
        completed = torch.cat([points, predicted_missing], dim=1)
        completed_feature = self.completed_point_encoder(completed).mean(1)
        feature = self.feature_head(torch.cat([observed_feature, completed_feature], dim=1))
        return self.class_head(feature), self.box_head(feature), predicted_missing

def completion_batch_from_rows(rows, observed_points=96):
    visible, hidden = [], []
    for index, row in enumerate(rows):
        full = row['points']
        rng = np.random.default_rng(index)
        order = np.argsort(full[:, 0] + rng.normal(0, 0.01, len(full)))
        visible.append(full[order[:observed_points]])
        hidden.append(full[order[observed_points:]])
    points = np.stack(visible).astype(np.float32)
    hidden = np.stack(hidden).astype(np.float32)
    labels = np.asarray([row['class_id'] for row in rows], dtype=np.int64)
    targets = np.stack([row['target'] for row in rows]).astype(np.float32)
    local = np.stack([fixed_knn(point, 8) for point in points])
    context = np.stack([fixed_knn(point, 16) for point in points])
    return (torch.tensor(points), torch.tensor(labels), torch.tensor(targets), torch.tensor(hidden), torch.tensor(local, dtype=torch.long), torch.tensor(context, dtype=torch.long))

def torch_chamfer(predicted, target):
    distances = torch.cdist(predicted, target)
    return distances.min(-1).values.mean() + distances.min(-2).values.mean()

print('completion-aware detector loaded: partial LiDAR -> predicted points -> graph features -> class/box heads')

completion-aware detector loaded: partial LiDAR -> predicted points -> graph features -> class/box heads


In [8]:
if DATA_AVAILABLE and DEVKIT_AVAILABLE and train_rows and val_rows:
    torch.manual_seed(7)
    model = CompletionAwareObjectPCRGNN(len(CLASS_NAMES))
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    train_points, train_labels, train_targets, train_hidden, train_local, train_context = completion_batch_from_rows(train_rows)
    model.train()
    for epoch in range(5):
        optimizer.zero_grad()
        logits, predicted, predicted_missing = model(train_points, train_local, train_context)
        loss = nn.functional.cross_entropy(logits, train_labels) + nn.functional.smooth_l1_loss(predicted, train_targets) + 0.25 * torch_chamfer(predicted_missing, train_hidden)
        loss.backward()
        optimizer.step()
        print(f'epoch {epoch + 1}/5 loss={loss.item():.4f}')
else:
    print('Training skipped: mount nuScenes mini and ensure both scene-disjoint object splits contain rows.')

Training skipped: mount nuScenes mini and ensure both scene-disjoint object splits contain rows.


## Evaluation process

After training, the model is switched to evaluation mode and run once on the held-out scenes. The notebook compares predicted and target centers, sizes, and yaw angles, computes axis-aligned 3D IoU, and reports the IoU hit rate at 0.25 as diagnostic precision/recall for this one-candidate-per-ground-truth crop experiment. It also prints per-class ground-truth counts, predictions, and IoU hit rates. These are development diagnostics, not official nuScenes mAP/NDS.

In [9]:
print('EVALUATION PROCESS')
print('1. Freeze the trained model and use only validation scenes')
print('2. Predict class, center offset, size, and yaw for each validation crop')
print('3. Compute center MAE, size MAE, yaw MAE, and axis-aligned 3D IoU')
print('4. Report IoU@0.25 diagnostic precision/recall and per-class hit rates')
if not (DATA_AVAILABLE and DEVKIT_AVAILABLE and train_rows and val_rows):
    print('STATUS: evaluation not run; no mounted nuScenes validation scenes are available.')

EVALUATION PROCESS
1. Freeze the trained model and use only validation scenes
2. Predict class, center offset, size, and yaw for each validation crop
3. Compute center MAE, size MAE, yaw MAE, and axis-aligned 3D IoU
4. Report IoU@0.25 diagnostic precision/recall and per-class hit rates
STATUS: evaluation not run; no mounted nuScenes validation scenes are available.


In [10]:
if DATA_AVAILABLE and DEVKIT_AVAILABLE and train_rows and val_rows:
    model.eval()
    with torch.no_grad():
        val_points, val_labels, val_targets, val_hidden, val_local, val_context = completion_batch_from_rows(val_rows)
        val_logits, val_predictions, val_missing = model(val_points, val_local, val_context)
    metrics = summarize_predictions(val_logits.numpy(), val_predictions.numpy(), val_labels.numpy(), val_targets.numpy())
    predicted_classes = val_logits.argmax(1).numpy()
    print('validation metrics (scene-disjoint nuScenes mini development split):')
    for key, value in metrics.items():
        print(f'{key}: {value:.4f}')
    print('per-class validation counts:')
    for class_id, name in enumerate(CLASS_NAMES):
        mask = val_labels.numpy() == class_id
        class_iou = axis_aligned_iou_3d(val_predictions.numpy()[mask, :3], np.maximum(val_predictions.numpy()[mask, 3:6], 1e-3), val_targets.numpy()[mask, :3], val_targets.numpy()[mask, 3:6]) if mask.any() else np.array([])
        class_hit_rate = float((class_iou >= 0.25).mean()) if len(class_iou) else 0.0
        print(name, 'ground truth:', int(mask.sum()), 'predicted:', int((predicted_classes == class_id).sum()), 'IoU@0.25:', round(class_hit_rate, 4))
else:
    print('Evaluation skipped because nuScenes mini is not available in this environment.')

Evaluation skipped because nuScenes mini is not available in this environment.


## Interpretation

The split is scene-disjoint and every crop retains its scene and box identifiers. The reported center, size, yaw, and IoU quantities are development diagnostics for this compact object-level experiment. The IoU implementation is axis-aligned to keep the notebook auditable; it must not be described as official oriented nuScenes detection AP. Full-scene proposal generation, tracking, calibration analysis, and official benchmark evaluation remain future work.